A.) **Dynamic Goal-Based Agent for Warehouse Logistics Optimization**.
A robotic agent operates in a warehouse modeled as an N×M grid environment. The agent starts at a predefined loading dock and must deliver packages to multiple destinations marked on the grid while avoiding dynamically placed obstacles.
Take suitable values of the following parameters.
- Warehouse dimensions: N×M grid size (M,N between 5 and 10, inclusive)
- Number of packages: P (between 2 and 6, inclusive)
- Number of obstacles: O (between 1 and 10, inclusive)
- Package locations: (X1, Y1), (X2, Y2), ... (XP, YP)
- Drop-off locations: (D1X, D1Y), (D2X, D2Y), ... (DPX, DPY)
- Robot starting position: S=(Sx,Sy), starts at a fixed cell but moves dynamically
- Movement cost: Each movement incurs a cost of 1 unit
- Delivery reward: Successfully delivering a package adds 10 units to the total reward - Obstacle penalty: Hitting an obstacle results in a (-5) penalty

**Note :** Packages locations and drop-off locations should not overlap.

**Q1. Represent the warehouse as an N×M matrix. Place the packages, drop-off points, and obstacles randomly. Display the initial warehouse configuration.**

In [ ]:
import numpy as np
import random
from collections import deque

# Setup the warehouse and return the values of grid and position of packages, delivery and obstacless
def setup_warehouse(robo_loc_x, robo_loc_y, rows, column, packages, obstacless,random_seed=42):
    """Initialize the warehouse environment with obstacles, packages, and drop-off points."""
    random.seed(random_seed)
    np.random.seed(random_seed)

    warehouse_grid = np.full((rows, column), '.', dtype=str)
    # mark Robot starting position in warehouse
    warehouse_grid[robo_loc_x, robo_loc_y] = 'R'

    # Randomly placing obstacles
    obstacle_positions = set()
    while len(obstacle_positions) < obstacless:
        obs_x, obs_y = random.randint(0, rows - 1), random.randint(0, column - 1)
        if (obs_x, obs_y) not in obstacle_positions:
            obstacle_positions.add((obs_x, obs_y))
            warehouse_grid[obs_x, obs_y] = 'X'

    # Assign packages and delivery locations
    package_locations = {}
    delivery_spots = {}

    while len(package_locations) < packages:
        pkg_x, pkg_y = random.randint(0, rows - 1), random.randint(0, column - 1)
        drop_x, drop_y = random.randint(0, rows - 1), random.randint(0, column - 1)

        # Check if locations are valid (not robot, obstacle, or existing package/delivery)
        if (pkg_x, pkg_y) != (robo_loc_x, robo_loc_y) and (drop_x, drop_y) != (robo_loc_x, robo_loc_y) and \
           (pkg_x, pkg_y) not in obstacle_positions and (drop_x, drop_y) not in obstacle_positions and \
           (pkg_x, pkg_y) not in delivery_spots and (drop_x, drop_y) not in package_locations and \
           (pkg_x, pkg_y) != (drop_x, drop_y):

            package_locations[(pkg_x, pkg_y)] = (drop_x, drop_y)
            delivery_spots[(drop_x, drop_y)] = (pkg_x, pkg_y)
            warehouse_grid[pkg_x, pkg_y] = 'P'
            warehouse_grid[drop_x, drop_y] = 'D'

    return warehouse_grid, package_locations, delivery_spots, obstacle_positions

# Display the warehouse using warehouse varible
def display_warehouse():
    for row in warehouse:
        print(" ".join(row))
    print()

# Assigned the calculated warehouse values required for traversing
def initialise_warehouse_values(robo_loc_x, robo_loc_y, rows, column, packages, obstacless):
    global warehouse, package_locations, drop_locations, obstacle_set, robot_position
    warehouse, package_locations, drop_locations, obstacle_set = setup_warehouse(robo_loc_x, robo_loc_y, rows, column,
                                                                                        packages, obstacless,
                                                                                        random_seed=42)
    robot_position = (robo_loc_x, robo_loc_y)
    print("Initial Warehouse Setup:")
    print()
    display_warehouse()

# Assigned the static values to the grid as mentioned in problem statement 1
def draw_static_warehouse():
  global rows, column, packages  # Warehouse rows, columns & total packages
  rows = 8
  column = 8
  packages = 5
  obstacless = 6
  robo_x = 0
  robo_y = 0
  initialise_warehouse_values(robo_x, robo_y, rows, column, packages, obstacless)

# Call the static warehouse method to display warehouse with static values
draw_static_warehouse()


Initial Warehouse Setup:

R X P D . . . .
X X . . . . . .
. . . . . . . .
. . X P . D . .
D . P X . . . .
. . . . . P . .
X . . P . D . .
. . . . D . . .



**Q2. Implement a goal-based agent that can identify all goals, plan a sequence of actions to reach the goal, use a search algorithm (BFS, DFS, or UCS) to find optimal paths, deliver all packages, and calculate the total cost.**

In [ ]:
# Implements Breadth-First Search (BFS) to determine the shortest path.
def bfs_search_path(start, destination, obstacles, rows, cols):
    move_directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    queue = deque([(start, [])])
    visited = set()
    while queue:
        (cur_x, cur_y), path_history = queue.popleft()

        if (cur_x, cur_y) in visited:
            continue
        visited.add((cur_x, cur_y))

        if (cur_x, cur_y) == destination:
            return path_history + [(cur_x, cur_y)], len(path_history)

        for move_x, move_y in move_directions:
            next_x, next_y = cur_x + move_x, cur_y + move_y

            if 0 <= next_x < rows and 0 <= next_y < cols :
                queue.append(((next_x, next_y), path_history + [(cur_x, cur_y)]))

    return None, float('inf')  # No possible path available

# Executes the utility-based agent that uses the bfs_search_path to pick and drop the packages
def execute_robot():
    total_movement_cost = 0
    delivery_score = 0
    obstacle_penalty = 0
    robot_location = robot_position
    print(f"Packages to be delivered: {packages}")
    print()
    for package_site, drop_site in package_locations.items():
        path_to_package, cost_to_package = bfs_search_path(robot_location, package_site, obstacle_set, rows,
                                                             column)
        path_to_delivery, cost_to_delivery = bfs_search_path(package_site, drop_site, obstacle_set, rows,
                                                               column)
        if path_to_package and path_to_delivery:
            print(f"Robot starts from {robot_location} to pick up the package at {package_site}.")
            print(f"Path taken for pickup: {path_to_package}")
            print(f"Robot needs to deliver the package from {package_site} to {drop_site}.")
            print(f"Path taken for delivery: {path_to_delivery}")
            print()
            total_movement_cost += (cost_to_package + cost_to_delivery)
            delivery_score += 10  # Reward for successful delivery
            robot_location = drop_site  # Update robot’s current position

            # Check if any obstacle is hit
            for step in path_to_package + path_to_delivery:
                if step in obstacle_set:
                    obstacle_penalty -= 5

    final_evaluation = delivery_score - total_movement_cost + obstacle_penalty
    print(f"Total Movement Cost: {total_movement_cost}")
    print(f"Total Delivery Reward: {delivery_score}")
    print(f"Obstacle Penalty: {obstacle_penalty}")
    print()
    print(f"Final Score: {final_evaluation}")


The warehouse is now initialized with predefined **static values**.
The delivery robot is ready to pick up and deliver packages using the Breadth-First Search (BFS) algorithm.

In [ ]:
# execute for static values
execute_robot()

Packages to be delivered: 5

Robot starts from (0, 0) to pick up the package at (3, 3).
Path taken for pickup: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]
Robot needs to deliver the package from (3, 3) to (0, 3).
Path taken for delivery: [(3, 3), (2, 3), (1, 3), (0, 3)]

Robot starts from (0, 3) to pick up the package at (6, 3).
Path taken for pickup: [(0, 3), (1, 3), (2, 3), (3, 3), (4, 3), (5, 3), (6, 3)]
Robot needs to deliver the package from (6, 3) to (7, 4).
Path taken for delivery: [(6, 3), (7, 3), (7, 4)]

Robot starts from (7, 4) to pick up the package at (0, 2).
Path taken for pickup: [(7, 4), (6, 4), (5, 4), (4, 4), (3, 4), (2, 4), (1, 4), (0, 4), (0, 3), (0, 2)]
Robot needs to deliver the package from (0, 2) to (6, 5).
Path taken for delivery: [(0, 2), (1, 2), (2, 2), (3, 2), (4, 2), (5, 2), (6, 2), (6, 3), (6, 4), (6, 5)]

Robot starts from (6, 5) to pick up the package at (4, 2).
Path taken for pickup: [(6, 5), (5, 5), (4, 5), (4, 4), (4, 3), (4, 2)]
Robot ne

**Q3. Choose a random seed value for the ease of reproducing the results. Your program should give outputs: the chosen path taken by the agent, total cost and rewards, final score based on penalties, movement costs, and successful deliveries.**

In [ ]:
# change the static values to dynamic as mentioned in problem statement 3
def draw_random_warehouse():
  global rows, column, packages
  rows = random.randint(5, 10)
  column = random.randint(5, 10)
  packages = random.randint(2, 6)
  obstacless = random.randint(1, 10)
  robo_x = random.randint(0, rows-1)
  robo_y = random.randint(0, column-1)
  initialise_warehouse_values(robo_x, robo_y, rows, column, packages, obstacless)

  print(f"Now warehouse values changed from static to random values")
  print(f"Here are the updated values")
  print()
  print(f"Warehouse grid -> {rows}*{column} grid")
  print(f"Package -> {packages}")
  print(f"Obstacless -> {obstacless}")
  print(f"Robot position -> {robot_position}")

# Call the static warehouse method to display warehouse with static values
draw_random_warehouse()

Initial Warehouse Setup:

P . D . . . . .
X X . D . . . .
. . . R . . . .
P . X . . . . D
. . . X . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . D . . P .
. . . . P . X .

Now warehouse values changed from static to random values
Here are the updated values

Warehouse grid -> 10*8 grid
Package -> 4
Obstacless -> 5
Robot position -> (2, 3)


In [ ]:
# execute for random values
execute_robot()

Packages to be delivered: 6

Robot starts from (6, 1) to pick up the package at (3, 2).
Path taken for pickup: [(6, 1), (5, 1), (4, 1), (3, 1), (3, 2)]
Robot needs to deliver the package from (3, 2) to (1, 1).
Path taken for delivery: [(3, 2), (2, 2), (1, 2), (1, 1)]

Robot starts from (1, 1) to pick up the package at (9, 6).
Path taken for pickup: [(1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (9, 2), (9, 3), (9, 4), (9, 5), (9, 6)]
Robot needs to deliver the package from (9, 6) to (0, 0).
Path taken for delivery: [(9, 6), (8, 6), (7, 6), (6, 6), (5, 6), (4, 6), (3, 6), (2, 6), (1, 6), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]

Robot starts from (0, 0) to pick up the package at (1, 3).
Path taken for pickup: [(0, 0), (1, 0), (1, 1), (1, 2), (1, 3)]
Robot needs to deliver the package from (1, 3) to (3, 0).
Path taken for delivery: [(1, 3), (2, 3), (3, 3), (3, 2), (3, 1), (3, 0)]

Robot starts from (3, 0) to pick up the package at (8, 3).
Path tak